# Project 5: dense embeddings for petroleum-reg-rag
Embeds all chunks with `BAAI/bge-small-en-v1.5` (via `fastembed`, ONNX, no PyTorch) and saves `embeddings.npy` to Google Drive.

**Before running:** put `all_chunks.json` in `MyDrive/petroleum-reg-rag/` on Google Drive.
CPU runtime is fine (a few seconds of compute).

In [ ]:
!pip -q install fastembed

## 1. Mount Drive and load chunks

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path

WORK = Path('/content/drive/MyDrive/petroleum-reg-rag')
WORK.mkdir(parents=True, exist_ok=True)
CHUNKS_PATH = WORK / 'all_chunks.json'
assert CHUNKS_PATH.exists(), f"Upload all_chunks.json to {WORK} first"

chunks = json.load(open(CHUNKS_PATH, encoding='utf-8'))
assert isinstance(chunks, list), f"Expected a list of chunks, got {type(chunks)}"
print("chunks:", len(chunks))
print("keys in first chunk:", list(chunks[0].keys()))
print(json.dumps(chunks[0], indent=2)[:800])

## 2. Choose what text to embed

In [ ]:
TEXT_KEY = next((k for k in ['text', 'content', 'body'] if k in chunks[0]), None)
CITE_KEY = next((k for k in ['citation', 'title', 'heading'] if k in chunks[0]), None)
assert TEXT_KEY, f"No text field found. Keys are: {list(chunks[0].keys())}. Set TEXT_KEY by hand."
print("TEXT_KEY =", TEXT_KEY, "| CITE_KEY =", CITE_KEY)

def embed_text(c):
    # Prepend the citation/heading so the vector also captures the section title
    if CITE_KEY and c.get(CITE_KEY):
        return f"{c[CITE_KEY]}\n{c[TEXT_KEY]}"
    return c[TEXT_KEY]

def label(c):
    return c.get(CITE_KEY) if CITE_KEY and c.get(CITE_KEY) else c[TEXT_KEY][:80]

texts = [embed_text(c) for c in chunks]
print(texts[0][:300])

## 3. Length check (bge-small truncates at 512 tokens)

In [ ]:
import numpy as np
words = np.array([len(t.split()) for t in texts])
print("median words:", int(np.median(words)), "| max words:", int(words.max()))
long_idx = np.where(words > 350)[0]
print("chunks likely to be truncated (>350 words):", len(long_idx))
for i in long_idx[:10]:
    print(" ", i, words[i], label(chunks[i]))

## 4. Embed (with a Drive checkpoint)

In [ ]:
import hashlib
from fastembed import TextEmbedding

MODEL = 'BAAI/bge-small-en-v1.5'
EMB_PATH = WORK / 'embeddings.npy'
META_PATH = WORK / 'embeddings_meta.json'
text_hash = hashlib.sha256('\n'.join(texts).encode('utf-8')).hexdigest()

model = TextEmbedding(MODEL)

reuse = False
if EMB_PATH.exists() and META_PATH.exists():
    meta = json.load(open(META_PATH))
    reuse = (meta.get('text_hash') == text_hash and meta.get('model') == MODEL)

if reuse:
    emb = np.load(EMB_PATH)
    print("Loaded existing embeddings from Drive")
else:
    emb = np.array(list(model.embed(texts, batch_size=32)), dtype=np.float32)
    emb /= np.linalg.norm(emb, axis=1, keepdims=True)   # unit vectors -> dot product = cosine
    np.save(EMB_PATH, emb)
    json.dump({
        'model': MODEL,
        'n_chunks': len(chunks),
        'dim': int(emb.shape[1]),
        'text_hash': text_hash,
        'row_order': 'row i of embeddings.npy = chunks[i] in all_chunks.json',
        'query_prefix': 'Represent this sentence for searching relevant passages: ',
    }, open(META_PATH, 'w'), indent=2)
    print("Saved to Drive")

print("shape:", emb.shape, "| norm check:", float(np.linalg.norm(emb[0])))
assert emb.shape[0] == len(chunks)

## 5. Sanity check: does dense retrieval fix BM25's misses?

In [ ]:
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

def search(q, k=5):
    qv = np.array(list(model.embed([QUERY_PREFIX + q])), dtype=np.float32)[0]
    qv /= np.linalg.norm(qv)
    sims = emb @ qv
    top = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i]), label(chunks[i])) for i in top]

tests = [
    "What is the penalty for gas flaring per 1,000 standard cubic feet?",
    "notify the Commission",   # the paraphrase-gap miss from BM25
]
for q in tests:
    print("\nQ:", q)
    for rank, (i, s, lab) in enumerate(search(q), 1):
        print(f"  {rank}. [{i}] {s:.3f}  {lab}")

## 6. Download the files, then commit them to `data/processed/`

In [ ]:
from google.colab import files
files.download(str(EMB_PATH))
files.download(str(META_PATH))